# Notebook 2 — From Business Documents to Searchable Knowledge

### Build an AI Business Copilot 🏔️ · Session 1

In **Notebook 1** we learned the key idea: an LLM doesn't know your business, so we
**ground** it by putting the right facts into the prompt. We did that by *pasting a policy
snippet in by hand*.

That doesn't scale. Trailhead has many documents and can't paste the right paragraph for
every customer question. In this notebook we build the piece that **finds the right
snippet automatically** — a searchable knowledge base over Trailhead's documents.

By the end you'll be able to:

1. Load Trailhead's policy documents (and see how a PDF becomes searchable text).
2. Explain why we split documents into **chunks**.
3. Turn text into **embeddings** — numbers that capture meaning.
4. Build a **FAISS** search index and retrieve the most relevant chunk for any question.

> 🐍 Run each cell (▶️) top to bottom. Read the notes above each cell. Look for ✏️ **Try
> it** markers where you can experiment. *This notebook makes no paid API calls* — it's
> all local search, so it's free to run as many times as you like.

## 1. Setup

This notebook uses the shared workshop files (the documents and a helper module called
`trailhead.py`). The cell below installs the libraries we need. The one after fetches the
workshop repo if it isn't already here.

In [ ]:
# Local embeddings + fast search + PDF reading. This takes a minute the first time.
%pip install -q sentence-transformers faiss-cpu pypdf pandas numpy

In [ ]:
# --- Make the workshop files available (works in Colab, Databricks, or locally) ---
import sys
import subprocess
from pathlib import Path

# If you're running this from Colab with just the notebook file, we clone the repo.
# Replace the URL below with the workshop's GitHub repository once it's published.
REPO_URL = "https://github.com/YOUR-ORG/trailhead-copilot"   # <-- ✏️ set this to the real repo URL

def find_repo_root():
    # Look upward from the current folder for the repo (data/ + documents/ + src/).
    for base in [Path.cwd(), *Path.cwd().parents]:
        if all((base / d).is_dir() for d in ("data", "documents", "src")):
            return base
    cloned = Path.cwd() / "trailhead-copilot"
    return cloned if (cloned / "src").is_dir() else None

root = find_repo_root()
if root is None:
    print("Workshop files not found — cloning the repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    root = find_repo_root()

if root is None:
    raise RuntimeError(
        "Could not find the workshop files. Set REPO_URL above to the real repository URL."
    )

sys.path.insert(0, str(root / "src"))   # so we can 'import trailhead'
print("Using workshop files at:", root)

## 2. Configuration

As in Notebook 1, the knobs you might tune live in one place:

- `CHUNK_MAX_CHARS` — how big each document chunk can be. Smaller chunks are more precise
  but hold less context; larger chunks hold more but can be less focused.
- `TOP_K` — how many chunks to retrieve for a question.
- `EMBED_MODEL_NAME` — the model that turns text into meaning-vectors. `all-MiniLM-L6-v2`
  is small, fast, free, and needs no API key.

In [ ]:
CHUNK_MAX_CHARS = 700
TOP_K = 4
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

## 3. Load the documents

Trailhead's knowledge lives in a handful of policy and FAQ documents. Let's load them and
see what we're working with. (`trailhead.py` just reads the files for us; we'll do the
interesting parts by hand.)

In [ ]:
import trailhead as th

documents = th.load_documents()   # a list of {"source": filename, "text": full text}

print("Loaded", len(documents), "documents:")
for doc in documents:
    print(f"  - {doc['source']:<22} ({len(doc['text'])} characters)")

In [ ]:
# Peek at the start of the return policy so we know what's inside.
return_doc = next(d for d in documents if d["source"] == "return_policy.md")
print(return_doc["text"][:600])

### A quick aside: how a PDF becomes text

Now that you've *read* the documents, here's how a copilot would handle them if they
arrived as **PDFs** (a very common business format). A PDF is really just formatted
pages — before we can search it, we have to **extract the plain text**. Libraries like
`pypdf` do exactly that.

Our workshop uses the clean markdown versions as the source of truth, but the cell below
shows the extraction step on one of the PDF copies so you've seen it.

In [ ]:
from pypdf import PdfReader

pdf_path = root / "documents" / "pdf" / "return_policy.pdf"
reader = PdfReader(str(pdf_path))

pdf_text = ""
for page in reader.pages:
    pdf_text += page.extract_text() + "\n"

print(f"Extracted {len(pdf_text)} characters from {pdf_path.name}\n")
print(pdf_text[:400])
# In practice you'd feed this extracted text into the same chunking step below. We'll use
# the tidy markdown so our examples stay clean.

## 4. Split the documents into chunks

Why not just search whole documents? Two reasons:

- **Precision.** A customer asking about *final sale returns* wants the one relevant
  paragraph, not the entire return policy. Smaller pieces = more focused answers.
- **Cost & focus.** Later we hand the retrieved text to Claude. Sending one tight
  paragraph is cheaper and less distracting than sending a whole document.

So we split each document into **chunks** — here, roughly paragraph-sized pieces. The
function below walks through a document's paragraphs and starts a new chunk whenever the
current one would get too big.

In [ ]:
def chunk_text(text, source, max_chars=CHUNK_MAX_CHARS):
    # Split on blank lines (paragraph boundaries) and pack paragraphs into chunks
    # that stay under max_chars. Each chunk remembers which document it came from.
    chunks, current = [], ""
    for paragraph in text.split("\n\n"):
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        if current and len(current) + len(paragraph) + 2 > max_chars:
            chunks.append({"source": source, "text": current.strip()})
            current = paragraph
        else:
            current = (current + "\n\n" + paragraph) if current else paragraph
    if current.strip():
        chunks.append({"source": source, "text": current.strip()})
    return chunks

# Build chunks for every document.
chunks = []
for doc in documents:
    chunks += chunk_text(doc["text"], doc["source"])

print("Created", len(chunks), "chunks from", len(documents), "documents.\n")
print("Example chunk (from", chunks[3]["source"] + "):\n")
print(chunks[3]["text"])

> ✏️ **Try it.** Change `CHUNK_MAX_CHARS` in the config cell to `300`, re-run the config
> cell and the chunking cell, and watch the number of chunks go up. Smaller chunks = more,
> tighter pieces. Set it back to `700` when you're done.

## 5. Turn text into meaning — embeddings

Here's the trick that makes search *understand* language. An **embedding** converts a
piece of text into a list of numbers (a vector) that captures its **meaning**. Text with
similar meaning gets similar numbers — even if the words are different.

Think of it like coordinates: *"How long do I have to return my boots?"* and *"What's the
deadline to send an item back?"* land close together, while *"best trail mix recipe"* lands
far away. We measure closeness with **cosine similarity** (1.0 = identical meaning, near 0
= unrelated).

Let's load the embedding model and see it in action.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer(EMBED_MODEL_NAME)   # downloads once, then cached

sentences = [
    "How long do I have to return my boots?",     # 0
    "What is the deadline to send an item back?",  # 1 — same meaning as 0
    "What's the best trail mix for a long hike?",  # 2 — unrelated
]
vectors = embedder.encode(sentences)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("Each sentence became a vector of length", len(vectors[0]))
print("similar meaning   (0 vs 1):", round(cosine(vectors[0], vectors[1]), 3))
print("unrelated meaning (0 vs 2):", round(cosine(vectors[0], vectors[2]), 3))

Notice the similar sentences score **much higher** than the unrelated pair — despite
sharing almost no words. That's semantic search: matching on *meaning*, not keywords. This
is why a customer can phrase a question any way they like and still find the right policy.

## 6. Build the search index (FAISS)

Now we embed **every chunk** and store the vectors in a **FAISS index** — a data structure
built for finding the nearest vectors fast. (With 25 chunks it's instant; the same code
scales to millions.)

We normalize the vectors first so that FAISS's inner-product search is exactly cosine
similarity — the same "closeness" score we just used.

In [ ]:
import faiss

chunk_texts = [c["text"] for c in chunks]
chunk_vectors = embedder.encode(chunk_texts)
chunk_vectors = np.asarray(chunk_vectors, dtype="float32")

faiss.normalize_L2(chunk_vectors)                 # so inner product == cosine similarity
index = faiss.IndexFlatIP(chunk_vectors.shape[1]) # an index over vectors of this length
index.add(chunk_vectors)

print("Indexed", index.ntotal, "chunks. The knowledge base is ready. 🔎")

## 7. Search the knowledge base

Retrieval works in three steps: embed the **question** the same way, ask FAISS for the
nearest chunk vectors, and return those chunks. Let's wrap that in a `search()` function
and try the exact question we had to paste by hand in Notebook 1.

In [ ]:
def search(query, k=TOP_K):
    q = embedder.encode([query])
    q = np.asarray(q, dtype="float32")
    faiss.normalize_L2(q)
    scores, ids = index.search(q, k)     # nearest k chunk vectors
    results = []
    for score, idx in zip(scores[0], ids[0]):
        results.append({"score": float(score), **chunks[idx]})
    return results

# The question we hand-answered in Notebook 1 — now retrieved automatically:
for hit in search("Can I return a final sale item?"):
    print(f"[{hit['score']:.2f}]  {hit['source']}")
    print("   " + hit["text"][:160].replace("\n", " ") + " ...\n")

🎯 The top result is the **Final Sale** paragraph from the return policy — found
automatically from a question that doesn't even contain the word "policy." No hand-pasting
required. That top chunk is exactly what we'll feed to Claude in the next notebook.

> ✏️ **Try it.** Run the cell below and swap in your own questions. Try wording a question
> the "wrong" way (slang, typos, synonyms) and see that semantic search still finds the
> right chunk.

In [ ]:
for query in [
    "how do I look after my tent",
    "does my backpack have a warranty",
    "whats the cost for overnight delivery",
]:
    top = search(query, k=1)[0]
    print(f"Q: {query}")
    print(f"   -> {top['source']}  [{top['score']:.2f}]")
    print(f"      {top['text'][:120].replace(chr(10), ' ')} ...\n")

## 8. What we built, and what's next

You built a **retriever**: give it a question, it returns the most relevant chunks of
Trailhead's documents. This is the "Retrieval" half of **RAG (Retrieval-Augmented
Generation)**.

Right now it just *finds* text. In **Notebook 3** we complete the loop: take the retrieved
chunks, hand them to Claude, and get back a clean, grounded, cited answer — turning this
retriever into a real document Q&A copilot.

> 💡 The same three functions you wrote here — `chunk_text`, embedding, and `search` — are
> packaged in `trailhead.py` (`build_chunks`, `build_index`, `search`) so later notebooks
> can reuse them without retyping.

## 9. Recap

- Pasting facts by hand doesn't scale — we need to **find** the right text automatically.
- We **chunk** documents into focused pieces.
- **Embeddings** turn text into vectors that capture *meaning*; similar meanings are close
  (measured by **cosine similarity**).
- A **FAISS** index finds the nearest chunks to a question fast.
- Result: a working **retriever** — the Retrieval in RAG.

## 10. Exercises

**Guided**
1. In the embeddings demo (Section 5), add a fourth sentence of your own and compute its
   cosine similarity to sentence 0. Predict whether it'll be high or low first.
2. Use `search()` to ask about **shipping time**. Which document and chunk comes back? Does
   the score make sense?

**Challenge**
3. Change `TOP_K` to `1` and then `6`. How does the number of returned chunks change what a
   downstream answer could "see"? What's the trade-off between too few and too many?
4. Add a brand-new fact to Trailhead by creating a short paragraph (e.g., a new "gift card
   policy") as a string, wrap it as a chunk `{"source": "gift_cards.md", "text": ...}`, add
   it to `chunks`, and **rebuild the index** (re-run Section 6). Then search for it. You've
   just updated the copilot's knowledge with no training — pure grounding.

## 11. Learn more

- **What is RAG? (Anthropic):** <https://docs.claude.com/en/docs/build-with-claude/embeddings>
- **Sentence-Transformers docs:** <https://www.sbert.net/>
- **FAISS (Meta AI) wiki:** <https://github.com/facebookresearch/faiss/wiki>
- **Codecademy — Intro to Generative AI:** <https://www.codecademy.com/learn/intro-to-generative-ai>
- **Codecademy — Retrieval-Augmented Generation (RAG):** <https://www.codecademy.com/resources/docs/ai/rag>

---
> ### 👩‍🏫 Instructor notes (remove before distributing to students)
>
> - **Timing (~55 min):** ~5 setup, ~10 load + PDF aside, ~10 chunking, ~12 embeddings
>   intuition (the cosine demo is the key "click"), ~8 index + search, remainder exercises.
> - **Free notebook:** no Claude calls here — reassure cost-conscious learners. The only
>   wait is the one-time model/deps download.
> - **The cosine demo (Section 5) is the concept anchor.** Emphasize: high score for
>   *different words, same meaning*; low for *related topic, different intent*. This is what
>   makes retrieval robust to how customers phrase things.
> - Tie Section 7 back to Notebook 1 explicitly: "the paragraph we pasted by hand is now
>   found automatically."
> - If `git clone` fails, REPO_URL isn't set — for a local run the finder locates the repo
>   without cloning, so live-teaching from the repo folder just works.
> - Databricks Free Edition: `faiss-cpu` and `sentence-transformers` install fine via the
>   `%pip` cell; the first import triggers the model download.